In [17]:
import dspy
import dspy.evaluate
import mlflow
import pandas as pd
from dotenv import find_dotenv, load_dotenv
from dspy import LM, Evaluate, Example, MIPROv2, ReAct

In [2]:
_ = load_dotenv(find_dotenv("../creds/.env"), verbose=True)

In [3]:
mlflow.set_experiment("dspy/03")
mlflow.dspy.autolog(
    log_evals=True,
    log_compiles=True,
    log_traces_from_compile=True,
)

2025/07/23 15:58:54 INFO mlflow.tracking.fluent: Experiment with name 'dspy/03' does not exist. Creating a new experiment.


In [5]:
def search_wikipedia(query: str) -> list[str]:
    results = dspy.ColBERTv2(url="http://20.102.90.50:2017/wiki17_abstracts")(query, k=3)

    return [x["text"] for x in results]

In [7]:
def create_question_example(row: pd.Series) -> Example:
    return Example(question=row["question"], answer=row["answer"]).with_inputs("question")

In [8]:
trainset_df = pd.read_json("../data/trainset.jsonl.gz", lines=True)
valset_df = pd.read_json("../data/valset.jsonl.gz", lines=True)

In [9]:
trainset = trainset_df.apply(create_question_example, axis=1).values.tolist()
valset = valset_df.apply(create_question_example, axis=1).values.tolist()

### qwen3:1.7b

In [4]:
lm = LM(
    "ollama_chat/qwen3:1.7b",
    api_base="http://localhost:11434",
    api_key="",
    max_tokens=40960,
    temperature=0.0,
    cache=False,
)
dspy.configure(lm=lm)

In [6]:
react = ReAct("question -> answer", tools=[search_wikipedia])

In [10]:
tp = MIPROv2(
    metric=dspy.evaluate.answer_exact_match,
    auto="light",
    num_threads=8,
)

In [11]:
# dspy.cache.load_memory_cache("../temp/memory_cache.pkl")

In [12]:
optimized_react = tp.compile(
    react,
    trainset=trainset[:10],
    valset=valset[:10],
    requires_permission_to_run=False,
)

2025/07/23 15:58:55 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'fe92b8138624478298b3dbcbd1d61b79', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current dspy workflow
2025/07/23 15:58:55 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 20
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 10

2025/07/23 15:58:55 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/07/23 15:58:55 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/07/23 15:58:55 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


100%|██████████| 10/10 [01:50<00:00, 11.09s/it]

Bootstrapped 1 full traces after 9 examples for up to 1 rounds, amounting to 10 attempts.


Bootstrapping set 4/6


...

2025/07/23 16:42:36 INFO dspy.evaluate.evaluate: Average Metric: 2 / 10 (20.0%)
2025/07/23 16:42:36 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 20.0 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 5', 'Predictor 1: Instruction 1', 'Predictor 1: Few-Shot Set 1'].
2025/07/23 16:42:36 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [30.0, 30.0, 30.0, 10.0, 10.0, 30.0, 0.0, 20.0, 20.0, 40.0, 30.0, 50.0, 50.0, 20.0, 30.0, 20.0, 30.0, 30.0, 30.0, 10.0, 20.0]
2025/07/23 16:42:36 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 50.0
2025/07/23 16:42:36 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/07/23 16:42:36 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 50.0!



🏃 View run eval_full_20 at: http://localhost:5000/#/experiments/1/runs/c1d723d8415443959b8ff2569bdb403b
🧪 View experiment at: http://localhost:5000/#/experiments/1


🏃 View run powerful-dog-97 at: http://localhost:5000/#/experiments/1/runs/fe92b8138624478298b3dbcbd1d61b79
🧪 View experiment at: http://localhost:5000/#/experiments/1


[Trace(trace_id=89d2b7580cf74260b71f2323008293d4), Trace(trace_id=de767efbe6cd4de0bf8d26c323bc428b), Trace(trace_id=39cde7d56ed4475cb557f41377500844), Trace(trace_id=9269a59b693a4898a020929142c28c07), Trace(trace_id=5235f728675d4f8fa4f6b9553ea16997), Trace(trace_id=e1206c2c921c4f688729b451f7db60df), Trace(trace_id=fe74adcb40ab45039a87dd57f3e022d6), Trace(trace_id=b47aa060526247539b45255ff32f372b), Trace(trace_id=3422484e415f4783beb1e80ffe7f7c53), Trace(trace_id=090125432316499992a97cceabd6e2a2)]

2025/07/23 16:46:49 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


In [18]:
evaluator = Evaluate(
    metric=dspy.evaluate.answer_exact_match,
    devset=valset,
    display_table=True,
    display_progress=True,
    num_threads=8,
)

In [21]:
original_score = evaluator(react)
original_score

Average Metric: 13.00 / 100 (13.0%): 100%|██████████| 100/100 [34:18<00:00, 20.59s/it]

2025/07/23 18:24:32 INFO dspy.evaluate.evaluate: Average Metric: 13 / 100 (13.0%)


,question,example_answer,trajectory,reasoning,pred_answer,answer_exact_match
0,"What movie did ""the king of cool"" play in with Bud Ekins as his st...","""The Great Escape""","{'thought_0': 'I need to search Wikipedia for the movie ""The King ...","The information from the observations confirms that ""The King of C...","The movie was ""The Great Escape.""",
1,whos family had their own reality tv show. Robert Kardashian or Ma...,their family reality television series,"{'thought_0': ""I need to confirm which of the two individuals' fam...",The information confirms that Robert Kardashian's family has their...,Robert Kardashian's family had their own reality TV show.,
2,Which star in Shadows in Paradise is a Russian ballerina?,Sofya Skya,{'thought_0': 'I need to search for information about the star in ...,The search results indicate that Sofya Skya is a Russian ballerina...,"Sofya Skya is the Russian ballerina star in ""Shadows in Paradise.""",
3,What was the meaning of the name of the man who appointed Amashsai?,comforter,"{'thought_0': 'The name ""Amashsai"" might refer to a historical fig...","The name ""Amashsai"" is associated with a biblical figure, Amashsai...","Amashsai was appointed by Nehemiah, a biblical figure who assigned...",
4,"In addition to the Austrian passport, what is needed to gain acces...",national identity card,{'thought_0': 'I need to determine what additional documents are r...,The Austrian passport allows visa-free access to 173 countries and...,No additional documents are needed beyond the Austrian passport fo...,
...,...,...,...,...,...,...
95,"What date did the American actress and singer-songwriter, known fo...","April 19, 1994",{'thought_0': 'I need to find the release date of the first album ...,The search results indicate conflicting information about the firs...,The American actress and singer-songwriter Joanna Lumley released ...,
96,What animated creatures were the title characters of the film whic...,seals,"{'thought_0': ""I need to determine which of Shakespeare's early pl...","The search results indicate that ""Shakespeare: The Animated Tales""...",There is no film based on a Shakespearean play from early in his c...,
97,The 1925 Saint Mary's Gaels football team represented what private...,Saint Mary's College of California,"{'thought_0': ""I need to determine which private, coeducational co...",The information from the observations confirms that the 1925 Saint...,"Saint Mary's College of California in Moraga, California.",
98,Were Dorothy Arzner and Richard Wallace both French film directors?,no,{'thought_0': 'I need to verify the nationalities of Dorothy Arzne...,The reasoning is based on verifying the nationalities of Dorothy A...,"No, Dorothy Arzner was American, and Richard Wallace was not Frenc...",


🏃 View run eval at: http://localhost:5000/#/experiments/1/runs/bd43f60cf99c4012b9d780f9204be10d
🧪 View experiment at: http://localhost:5000/#/experiments/1


13.0

[Trace(trace_id=471f08a902244058b60780e86e7536be), Trace(trace_id=81a98f47e0654c5fafc1504178e2e90e), Trace(trace_id=3a1063fa75f94f489158c745351e6504), Trace(trace_id=3b3681f1974c4797862846986e3efdf6), Trace(trace_id=2c215893afa24e78ad202d6f7b6423ed), Trace(trace_id=305c4694b33f43c783b374ee81731f69), Trace(trace_id=471afbd5bb464585a9b2c25be6db1bb7), Trace(trace_id=7eed1891a0744ffda85b442be20c7f65), Trace(trace_id=dcea82abf82d4fbb9e037d85a023e61b), Trace(trace_id=e37fc55b9f5942b88e80502cbd2bbcb1)]

In [22]:
optimized_score = evaluator(optimized_react)
optimized_score

Average Metric: 36.00 / 100 (36.0%): 100%|██████████| 100/100 [15:36<00:00,  9.37s/it]

2025/07/23 18:45:45 INFO dspy.evaluate.evaluate: Average Metric: 36 / 100 (36.0%)


,question,example_answer,trajectory,reasoning,pred_answer,answer_exact_match
0,"What movie did ""the king of cool"" play in with Bud Ekins as his st...","""The Great Escape""","{'thought_0': 'I need to find out which movie ""The King of Cool"" s...",The information from the observations indicates that Bud Ekins was...,The Great Escape,✔️ [True]
1,whos family had their own reality tv show. Robert Kardashian or Ma...,their family reality television series,"{'thought_0': ""I need to confirm which of Robert Kardashian or Man...",The information from the observations confirms that Robert Kardash...,Robert Kardashian's family,
2,Which star in Shadows in Paradise is a Russian ballerina?,Sofya Skya,{'thought_0': 'I need to search for information about the star in ...,"The search results for ""Shadows in Paradise"" cast members do not i...","There is no Russian ballerina in the cast of ""Shadows in Paradise.""",
3,What was the meaning of the name of the man who appointed Amashsai?,comforter,{'thought_0': 'I need to determine who the man who appointed Amash...,"The name Nehemiah means ""Yahweh comforts"" or ""Comforter,"" as deriv...","Nehemiah means ""Yahweh comforts"" or ""Comforter.""",
4,"In addition to the Austrian passport, what is needed to gain acces...",national identity card,{'thought_0': 'I need to determine what additional documents are r...,The information from the observations indicates that Austrian citi...,No additional documents are needed beyond the Austrian passport.,
...,...,...,...,...,...,...
95,"What date did the American actress and singer-songwriter, known fo...","April 19, 1994","{'thought_0': ""I need to search Wikipedia to find the release date...",The information from the observations confirms that Barbara Eden's...,1984,
96,What animated creatures were the title characters of the film whic...,seals,{'thought_0': 'I need to confirm which animated creature is from t...,"The search results indicate that ""The Journey to Melonia"" is the f...",The animated creature's name is not explicitly mentioned in the pr...,
97,The 1925 Saint Mary's Gaels football team represented what private...,Saint Mary's College of California,"{'thought_0': ""I need to search Wikipedia for information about th...",The information from the observations confirms that the 1925 Saint...,Saint Mary's College of California,✔️ [True]
98,Were Dorothy Arzner and Richard Wallace both French film directors?,no,{'thought_0': 'I need to verify the nationalities of Dorothy Arzne...,The information from the observations indicates that Dorothy Arzne...,no,✔️ [True]


🏃 View run eval at: http://localhost:5000/#/experiments/1/runs/589f0ddc2af243d1967d6b8e4e3900dd
🧪 View experiment at: http://localhost:5000/#/experiments/1


36.0

[Trace(trace_id=f5919f1ee87d418b85edc86b1e56c29f), Trace(trace_id=1c1d939229be4f44b2665a058d797e4e), Trace(trace_id=67da43d94e39482581e6c58309ea9684), Trace(trace_id=f655c1ecb6fd4ee89b9df80291ac0d6c), Trace(trace_id=89517f1d322740f48e153d4cbb0467ce), Trace(trace_id=710114264901422090a5c59928ccf239), Trace(trace_id=0175a364bb2549be91f298002d140cb1), Trace(trace_id=a06d341624294684a0d0ef43b3ee87bd), Trace(trace_id=a3e8b4667fbe44f380c2124680629777), Trace(trace_id=facf78dd0a2741e7952a6f33ee6eaf05)]

### qwen3:4b

In [33]:
lm = LM(
    "ollama_chat/qwen3:4b",
    api_base="http://localhost:11434",
    api_key="",
    max_tokens=40960,
    temperature=0.0,
    cache=False,
)
dspy.configure(lm=lm)

In [34]:
react = ReAct("question -> answer", tools=[search_wikipedia])

In [35]:
tp = MIPROv2(
    metric=dspy.evaluate.answer_exact_match,
    auto="light",
    num_threads=8,
)

In [36]:
optimized_react = tp.compile(
    react,
    trainset=trainset[:10],
    valset=valset[:10],
    requires_permission_to_run=False,
)

2025/07/23 19:40:04 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '8ce79430ad024094b1e8dc1a7e5b7b37', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current dspy workflow
2025/07/23 19:40:04 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 20
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 10

2025/07/23 19:40:04 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/07/23 19:40:04 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/07/23 19:40:04 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


100%|██████████| 10/10 [13:51<00:00, 83.15s/it] 

Bootstrapped 2 full traces after 9 examples for up to 1 rounds, amounting to 10 attempts.


Bootstrapping set 4/6


...


🏃 View run eval_full_19 at: http://localhost:5000/#/experiments/1/runs/8f38c44dfa494f5cafa2b557de3f0458
🧪 View experiment at: http://localhost:5000/#/experiments/1
Average Metric: 4.00 / 10 (40.0%): 100%|██████████| 10/10 [02:52<00:00, 17.23s/it]

2025/07/23 22:23:14 INFO dspy.evaluate.evaluate: Average Metric: 4 / 10 (40.0%)
2025/07/23 22:23:14 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 40.0 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 0', 'Predictor 1: Instruction 2', 'Predictor 1: Few-Shot Set 1'].
2025/07/23 22:23:14 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [0.0, 30.0, 50.0, 30.0, 50.0, 50.0, 20.0, 60.0, 30.0, 60.0, 40.0, 50.0, 40.0, 30.0, 50.0, 50.0, 40.0, 60.0, 40.0, 60.0, 40.0]
2025/07/23 22:23:14 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 60.0
2025/07/23 22:23:14 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/07/23 22:23:14 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 60.0!



🏃 View run eval_full_20 at: http://localhost:5000/#/experiments/1/runs/419ffb6e53e1446b8e42d00afac603c8
🧪 View experiment at: http://localhost:5000/#/experiments/1


🏃 View run luminous-jay-870 at: http://localhost:5000/#/experiments/1/runs/8ce79430ad024094b1e8dc1a7e5b7b37
🧪 View experiment at: http://localhost:5000/#/experiments/1


[Trace(trace_id=a4458a0bbbf043e9b2592b9197a2d255), Trace(trace_id=3d72a4247e13414cb742245d79cd566e), Trace(trace_id=681b40298d954dc5b8a9f88df8bde431), Trace(trace_id=e5785b1ecfcd46ba8b19496265f0e445), Trace(trace_id=b8cd4172da694c8a85f27a271df47981), Trace(trace_id=ce16ed443f87423c90c5aa566a8b7083), Trace(trace_id=ae849dd9e5a44536b5db2811dab98465), Trace(trace_id=1130ed837ad24dddb020dc1f916030cf), Trace(trace_id=f300cb8f4cc74a9bb669a37cde4b73c1), Trace(trace_id=d44dcb9fbc7f4848821268e9d719e19d)]

In [37]:
evaluator = Evaluate(
    metric=dspy.evaluate.answer_exact_match,
    devset=valset,
    display_table=True,
    display_progress=True,
    num_threads=8,
)

In [38]:
original_score = evaluator(react)
original_score

Average Metric: 26.00 / 100 (26.0%): 100%|██████████| 100/100 [51:00<00:00, 30.61s/it]

2025/07/23 23:14:15 INFO dspy.evaluate.evaluate: Average Metric: 26 / 100 (26.0%)


,question,example_answer,trajectory,reasoning,pred_answer,answer_exact_match
0,"What movie did ""the king of cool"" play in with Bud Ekins as his st...","""The Great Escape""","{'thought_0': 'I need to determine which movie features ""the king ...",The search for Bud Ekins' stunt double movies identified films lik...,There is no movie mentioned in the provided information where Pete...,
1,whos family had their own reality tv show. Robert Kardashian or Ma...,their family reality television series,"{'thought_0': 'I need to determine which family, Robert Kardashian...",The search results confirm that Robert Kardashian's family is asso...,"Robert Kardashian's family had their own reality TV show, ""Keeping...",
2,Which star in Shadows in Paradise is a Russian ballerina?,Sofya Skya,"{'thought_0': 'The question asks about a specific character in ""Sh...","The question refers to ""Shadows in Paradise,"" which has two distin...","The provided information does not specify a Russian ballerina in ""...",
3,What was the meaning of the name of the man who appointed Amashsai?,comforter,{'thought_0': 'To determine the meaning of the name of the man who...,"The man who appointed Amashsai is Nehemiah, as established through...","The meaning of the name Nehemiah is ""Yahweh comforts"" or ""comforter.""",
4,"In addition to the Austrian passport, what is needed to gain acces...",national identity card,"{'thought_0': ""To determine what additional documents are required...",The observation from the Wikipedia search confirms that Austrian c...,No additional documents are required beyond the Austrian passport ...,
...,...,...,...,...,...,...
95,"What date did the American actress and singer-songwriter, known fo...","April 19, 1994",{'thought_0': 'To determine the release date of the first album by...,The actress associated with the role of Peggy Bundy is Katey Sagal...,The American actress and singer-songwriter known for her role as P...,
96,What animated creatures were the title characters of the film whic...,seals,"{'thought_0': ""The question asks for animated creatures that were ...","The film ""Gnomeo & Juliet"" (2011) is based on Shakespeare's ""Romeo...",Romeo and Juliet,
97,The 1925 Saint Mary's Gaels football team represented what private...,Saint Mary's College of California,"{'thought_0': 'The question asks for the private, coeducational co...",The search results confirm that the 1925 Saint Mary's Gaels footba...,Saint Mary's College of California,✔️ [True]
98,Were Dorothy Arzner and Richard Wallace both French film directors?,no,"{'thought_0': ""I need to verify the nationalities of Dorothy Arzne...","Dorothy Arzner was confirmed to be an American film director, not ...","No, Dorothy Arzner was an American film director, and Richard Wall...",


🏃 View run eval at: http://localhost:5000/#/experiments/1/runs/d0d073789fe045d28abc4a2303bb528f
🧪 View experiment at: http://localhost:5000/#/experiments/1


26.0

[Trace(trace_id=14e2c163770d4ce08dee1d56e3be8d26), Trace(trace_id=38f73f4681844e76a12322343ef0bd42), Trace(trace_id=b60b8bef35eb41289e5e9a041604470b), Trace(trace_id=af0632839d7a4cb1a378f7d72dd20a23), Trace(trace_id=691eb5d98b1e4f5dafdbb79644191f49), Trace(trace_id=ff767eaa0da349688e79b9b835172ab4), Trace(trace_id=d8757c7ea51146c98eb799ca2b434f5f), Trace(trace_id=5d1d49bbdf7b4e649f55922e257e8c56), Trace(trace_id=6fe5627b764a4be2a716137511d177fd), Trace(trace_id=53741e2e62794a01ab04bbdfdd393f7f)]

In [39]:
optimized_score = evaluator(optimized_react)
optimized_score

Average Metric: 52.00 / 100 (52.0%): : 101it [1:08:54, 40.94s/it]                        

2025/07/24 00:23:10 INFO dspy.evaluate.evaluate: Average Metric: 52 / 100 (52.0%)


,question,example_answer,trajectory,reasoning,pred_answer,answer_exact_match
0,"What movie did ""the king of cool"" play in with Bud Ekins as his st...","""The Great Escape""","{'thought_0': 'I need to determine which movie features ""the king ...","The term ""the king of cool"" refers to Steve McQueen, a renowned ac...",The Great Escape,✔️ [True]
1,whos family had their own reality tv show. Robert Kardashian or Ma...,their family reality television series,"{'thought_0': ""I need to determine which family, Robert Kardashian...",The search results indicate that Robert Kardashian's family is ass...,Robert Kardashian,
2,Which star in Shadows in Paradise is a Russian ballerina?,Sofya Skya,"{'thought_0': 'I need to identify the Russian ballerina in ""Shadow...","The 2010 American film ""Shadows in Paradise"" features Sofya Skya a...",Sofya Skya,✔️ [True]
3,What was the meaning of the name of the man who appointed Amashsai?,comforter,{'thought_0': 'I need to determine who appointed Amashsai and the ...,The search results indicate that Amashsai was appointed by Nehemia...,"The name Nehemiah means ""Yahweh comforts"" or ""comforter.""",
4,"In addition to the Austrian passport, what is needed to gain acces...",national identity card,{'thought_0': 'I need to determine the additional requirements for...,The Austrian passport itself grants visa-free or visa-on-arrival a...,No additional documents are required beyond the Austrian passport.,
...,...,...,...,...,...,...
95,"What date did the American actress and singer-songwriter, known fo...","April 19, 1994",{'thought_0': 'I need to identify the American actress and singer-...,The American actress known for playing Peggy Bundy is Katey Sagal....,"April 19, 1994",✔️ [True]
96,What animated creatures were the title characters of the film whic...,seals,"{'thought_0': ""I need to find out which animated creatures were th...",The question asks for animated creatures as title characters of a ...,Romeo and Juliet,
97,The 1925 Saint Mary's Gaels football team represented what private...,Saint Mary's College of California,{'thought_0': 'I need to confirm that the 1925 Saint Mary\'s Gaels...,The 1925 Saint Mary's Gaels football team represented Saint Mary's...,Saint Mary's College of California,✔️ [True]
98,Were Dorothy Arzner and Richard Wallace both French film directors?,no,{'thought_0': 'I need to verify the nationalities and backgrounds ...,"Dorothy Arzner was an American film director, not French. Richard ...",no,✔️ [True]


🏃 View run eval at: http://localhost:5000/#/experiments/1/runs/d47ed21946fe4125b5ae53ff403786b7
🧪 View experiment at: http://localhost:5000/#/experiments/1


52.0

[Trace(trace_id=e9f8e745a8294c91b86bdc456a89b871), Trace(trace_id=3da6f947280e4dce9ffa0834b16274e5), Trace(trace_id=a7aee9199b6b496dabde3aaf10ae474c), Trace(trace_id=c5cb93089ca542118c549860f907e2c4), Trace(trace_id=c314912c1b214198a432fe05f74efdd0), Trace(trace_id=453ccf7aa7ab4e33b864ef65b24c4191), Trace(trace_id=099a37793ea94721be5afd8cb27034e5), Trace(trace_id=7653fff3dea7403eb4973a5d2ca19d78), Trace(trace_id=add010e42c3740c6a586b796d99cb28c), Trace(trace_id=bf58cfb963ad4fd1a9cf0414269de595)]

### gemma3:4b

In [40]:
lm = LM(
    "ollama_chat/gemma3:4b",
    api_base="http://localhost:11434",
    api_key="",
    max_tokens=40960,
    temperature=0.0,
    cache=False,
)
dspy.configure(lm=lm)

In [41]:
react = ReAct("question -> answer", tools=[search_wikipedia])

In [42]:
tp = MIPROv2(
    metric=dspy.evaluate.answer_exact_match,
    auto="light",
    num_threads=8,
)

In [43]:
optimized_react = tp.compile(
    react,
    trainset=trainset[:10],
    valset=valset[:10],
    requires_permission_to_run=False,
)

2025/07/24 00:23:11 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'f977d1c7c55943c7becbe6e5fa9013a9', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current dspy workflow
2025/07/24 00:23:11 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 20
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 10

2025/07/24 00:23:11 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/07/24 00:23:11 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/07/24 00:23:11 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


100%|██████████| 10/10 [02:52<00:00, 17.23s/it]

Bootstrapped 4 full traces after 9 examples for up to 1 rounds, amounting to 10 attempts.


Bootstrapping set 4/6


...

Average Metric: 2.00 / 10 (20.0%): 100%|██████████| 10/10 [11:02<00:00, 66.21s/it] 

2025/07/24 01:46:33 INFO dspy.evaluate.evaluate: Average Metric: 2 / 10 (20.0%)
2025/07/24 01:46:33 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 20.0 with parameters ['Predictor 0: Instruction 1', 'Predictor 0: Few-Shot Set 1', 'Predictor 1: Instruction 1', 'Predictor 1: Few-Shot Set 5'].
2025/07/24 01:46:33 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [30.0, 10.0, 30.0, 30.0, 40.0, 0.0, 20.0, 40.0, 40.0, 40.0, 30.0, 40.0, 0.0, 20.0, 20.0, 0.0, 40.0, 30.0, 10.0, 30.0, 20.0]
2025/07/24 01:46:33 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 40.0
2025/07/24 01:46:33 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/07/24 01:46:33 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 40.0!



🏃 View run eval_full_20 at: http://localhost:5000/#/experiments/1/runs/5f6aa0e49d764113b7e4eedc2bc65446
🧪 View experiment at: http://localhost:5000/#/experiments/1


🏃 View run grandiose-elk-615 at: http://localhost:5000/#/experiments/1/runs/f977d1c7c55943c7becbe6e5fa9013a9
🧪 View experiment at: http://localhost:5000/#/experiments/1


[Trace(trace_id=0dec25035eb7434e82bc3791c093ab6a), Trace(trace_id=b4f07acbc4a9423c90b644b31f3eb844), Trace(trace_id=48800bf76c3041b392c533efd3e2c692), Trace(trace_id=0e4c07cb8f8b435698ad479cc9a1c2e5), Trace(trace_id=916bba21c2164e769d7d6e3513ef5add), Trace(trace_id=34d0d4562f0546188f10b9e481deeba5), Trace(trace_id=f2d22809c2a643d4a5321f366e9587f0), Trace(trace_id=d1af3e104379432f84fc423f557f87f2), Trace(trace_id=d0bdfd9bf54a428bb8def36d1cd455d9), Trace(trace_id=cf95613e876c4d8dbd3125f1ace763e3)]

In [44]:
evaluator = Evaluate(
    metric=dspy.evaluate.answer_exact_match,
    devset=valset,
    display_table=True,
    display_progress=True,
    num_threads=8,
)

In [45]:
original_score = evaluator(react)
original_score

Average Metric: 29.00 / 100 (29.0%): 100%|██████████| 100/100 [26:35<00:00, 15.95s/it]

2025/07/24 02:13:09 INFO dspy.evaluate.evaluate: Average Metric: 29 / 100 (29.0%)


,question,example_answer,trajectory,reasoning,pred_answer,answer_exact_match
0,"What movie did ""the king of cool"" play in with Bud Ekins as his st...","""The Great Escape""","{'thought_0': 'Okay, I need to find out which movie Steve McQueen ...","Okay, the answer.",The Great Escape.,✔️ [True]
1,whos family had their own reality tv show. Robert Kardashian or Ma...,their family reality television series,"{'thought_0': ""Okay, I need to find out which of Robert Kardashian...",The Wikipedia page for Robert Kardashian states that he appeared o...,Robert Kardashian,
2,Which star in Shadows in Paradise is a Russian ballerina?,Sofya Skya,"{'thought_0': 'Okay, I need to find out which star in Shadows in P...",I have conducted several searches on Wikipedia to determine which ...,I cannot answer this question.,
3,What was the meaning of the name of the man who appointed Amashsai?,comforter,"{'thought_0': 'Okay, the question is about the meaning of the name...",The question asks for the meaning of the name of the man who appoi...,The name Amashsai (or Amashai) was likely derived from the Hebrew ...,
4,"In addition to the Austrian passport, what is needed to gain acces...",national identity card,"{'thought_0': 'Okay, the question is asking about what is needed t...",The question asks what is needed to gain access to 173 countries a...,A Home Office travel document or a Bruneian International Certific...,
...,...,...,...,...,...,...
95,"What date did the American actress and singer-songwriter, known fo...","April 19, 1994","{'thought_0': ""Okay, I need to find the release date of the actres...",I have successfully identified the correct Elisabeth – Elisabeth W...,"September 14, 2010",
96,What animated creatures were the title characters of the film whic...,seals,"{'thought_0': 'Okay, the question is about animated creatures that...",The question asks for the animated creatures that were the title c...,Seals,✔️ [True]
97,The 1925 Saint Mary's Gaels football team represented what private...,Saint Mary's College of California,"{'thought_0': ""Okay, I need to find out which college the 1925 Sai...",The provided Wikipedia entries clearly state that the 1925 Saint M...,Saint Mary's College of California,✔️ [True]
98,Were Dorothy Arzner and Richard Wallace both French film directors?,no,{'thought_0': 'This is a factual question about film directors. I ...,I investigated the biographical information for both Dorothy Arzne...,No.,✔️ [True]


🏃 View run eval at: http://localhost:5000/#/experiments/1/runs/eabfea999baf4da8956cdb86af4514fe
🧪 View experiment at: http://localhost:5000/#/experiments/1


29.0

[Trace(trace_id=357f2aee1d574442a5fae5435ed1b83e), Trace(trace_id=b632132a3d6b4754a4a38b7debf47a70), Trace(trace_id=5728757af526446f91b4ce0f5cf5a846), Trace(trace_id=27127ba167c44af897cb9313fb5b8154), Trace(trace_id=d9eb2eb7c9fa4e948e2b97448f2b16f6), Trace(trace_id=75a15dc82c134983b8719a446a036aa1), Trace(trace_id=71371148c325492db325734c1481c4f7), Trace(trace_id=caccc19dfe1e4bb19586b425c8508c3f), Trace(trace_id=456ab2bdd50a4bb3af9b165ad9da4c86), Trace(trace_id=53853a389db14f7e864e4ca384fa9dd1)]

In [46]:
optimized_score = evaluator(optimized_react)
optimized_score

Average Metric: 2.00 / 9 (22.2%):   9%|▉         | 9/100 [05:16<45:46, 30.18s/it]    

2025/07/24 02:18:34 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/07/24 02:18:35 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 7.00 / 38 (18.4%):  38%|███▊      | 38/100 [22:52<1:38:16, 95.11s/it] 

2025/07/24 02:36:06 ERROR dspy.utils.parallelizer: Error for Example({'question': 'What American singer born 1951 does Jim Rooney  have credits for?', 'answer': 'Iris Luella DeMent'}) (input_keys={'question'}): Adapter JSONAdapter failed to parse the LM response. 

LM Response: { "_,_ 1_ _ _ _ _ _ __ __ __ __ __ __ __ __ __ __ __ __ _ 

Expected to find output fields in the LM response: [next_thought, next_tool_name, next_tool_args] 

Actual output fields parsed from the LM response: [] 

. Set `provide_traceback=True` for traceback.


Average Metric: 11.00 / 48 (22.9%):  49%|████▉     | 49/100 [25:35<12:12, 14.36s/it] 

...

2025/07/24 03:27:21 INFO dspy.evaluate.evaluate: Average Metric: 24.0 / 100 (24.0%)


,question,example_answer,trajectory,reasoning,pred_answer,answer_exact_match,answer
0,"What movie did ""the king of cool"" play in with Bud Ekins as his st...","""The Great Escape""",{'thought_0': 'I need to find out which movie Steve McQueen starre...,"Based on my investigation, Steve McQueen starred in the movie ""Bul...",Bullitt,,NaN
1,whos family had their own reality tv show. Robert Kardashian or Ma...,their family reality television series,{'thought_0': 'I need to find out which of Robert Kardashian or Ma...,"Robert Kardashian had his own reality TV show, ""Keeping Up with th...",Robert Kardashian,,NaN
2,Which star in Shadows in Paradise is a Russian ballerina?,Sofya Skya,"{'thought_0': 'I need to identify which character in ""Shadows in P...","Based on my investigation, Sofya Skya plays a role in ""Shadows in ...",Sofya Skya,✔️ [True],NaN
3,What was the meaning of the name of the man who appointed Amashsai?,comforter,{'thought_0': 'I need to find out the meaning behind the name of t...,The Wikipedia search reveals that the name Amashsai (also spelled ...,burdensome,,NaN
4,"In addition to the Austrian passport, what is needed to gain acces...",national identity card,{'thought_0': 'I need to find out what is needed to gain access to...,"Based on the provided information, a British passport (and therefo...",Austrian passport,,NaN
...,...,...,...,...,...,...,...
95,"What date did the American actress and singer-songwriter, known fo...","April 19, 1994",{'thought_0': 'I need to find the release date of the actress know...,I have repeatedly confirmed that Elisabeth Withers released her se...,"September 14, 2010",,NaN
96,What animated creatures were the title characters of the film whic...,seals,"{'thought_0': ""The question refers to a film based on one of Shake...","Based on the extensive Wikipedia search results, the title charact...",puppets,,NaN
97,The 1925 Saint Mary's Gaels football team represented what private...,Saint Mary's College of California,"{'thought_0': ""I need to find out which college the 1925 Saint Mar...",I've analyzed the provided text and identified the key information...,Saint Mary’s College of California,,NaN
98,Were Dorothy Arzner and Richard Wallace both French film directors?,no,{'thought_0': 'I need to find out if Dorothy Arzner and Richard Wa...,The Wikipedia search revealed that Dorothy Arzner was an American ...,no,✔️ [True],NaN


🏃 View run eval at: http://localhost:5000/#/experiments/1/runs/f64f82b74ab14b6eaaaedab76e20d635
🧪 View experiment at: http://localhost:5000/#/experiments/1


24.0

[Trace(trace_id=cfff9844a7984e8a8122b2ca6b358bf1), Trace(trace_id=9c597f082f294feda0054e225c27daf6), Trace(trace_id=fa99f3fafdfb4db7893ebafa23aa3fe3), Trace(trace_id=118917bfb4ae480d8508604551f98702), Trace(trace_id=6499be218a614deead94fd4b4a3369f9), Trace(trace_id=e4f16806da294b09b24e8b79e0c097e2), Trace(trace_id=145b840b26ac471b96aeb2d144e9e0a6), Trace(trace_id=9cbe1d365d6f4ccdaca2ed5ddd51f7a0), Trace(trace_id=67dbe3f321fa4661bbd10f772e028337), Trace(trace_id=90a6ff1ea22c411f9a980a5c3336490a)]

### llama3.1:8b

In [47]:
lm = LM(
    "ollama_chat/llama3.1:8b",
    api_base="http://localhost:11434",
    api_key="",
    max_tokens=40960,
    temperature=0.0,
    cache=False,
)
dspy.configure(lm=lm)

In [48]:
react = ReAct("question -> answer", tools=[search_wikipedia])

In [49]:
tp = MIPROv2(
    metric=dspy.evaluate.answer_exact_match,
    auto="light",
    num_threads=8,
)

In [50]:
optimized_react = tp.compile(
    react,
    trainset=trainset[:10],
    valset=valset[:10],
    requires_permission_to_run=False,
)

2025/07/24 03:27:21 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'ff32884fbdd94eeb9336508483637e8a', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current dspy workflow
2025/07/24 03:27:21 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 20
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 10

2025/07/24 03:27:21 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/07/24 03:27:21 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/07/24 03:27:21 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


100%|██████████| 10/10 [03:40<00:00, 22.05s/it]

Bootstrapped 3 full traces after 9 examples for up to 1 rounds, amounting to 10 attempts.


Bootstrapping set 4/6


...


🏃 View run eval_full_19 at: http://localhost:5000/#/experiments/1/runs/4edb309665874f20b9c0a2c2522ea840
🧪 View experiment at: http://localhost:5000/#/experiments/1
Average Metric: 4.00 / 10 (40.0%): 100%|██████████| 10/10 [03:34<00:00, 21.50s/it]

2025/07/24 05:29:29 INFO dspy.evaluate.evaluate: Average Metric: 4 / 10 (40.0%)
2025/07/24 05:29:30 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 40.0 with parameters ['Predictor 0: Instruction 1', 'Predictor 0: Few-Shot Set 1', 'Predictor 1: Instruction 1', 'Predictor 1: Few-Shot Set 5'].
2025/07/24 05:29:30 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [10.0, 20.0, 40.0, 10.0, 30.0, 50.0, 20.0, 30.0, 20.0, 10.0, 30.0, 30.0, 40.0, 40.0, 20.0, 40.0, 30.0, 60.0, 30.0, 30.0, 40.0]
2025/07/24 05:29:30 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 60.0
2025/07/24 05:29:30 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/07/24 05:29:30 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 60.0!



🏃 View run eval_full_20 at: http://localhost:5000/#/experiments/1/runs/a83fe317912744889363e87a778e8d3c
🧪 View experiment at: http://localhost:5000/#/experiments/1


🏃 View run flawless-vole-925 at: http://localhost:5000/#/experiments/1/runs/ff32884fbdd94eeb9336508483637e8a
🧪 View experiment at: http://localhost:5000/#/experiments/1


[Trace(trace_id=e9cb32b69e0a43aaadba40aab237d672), Trace(trace_id=1f3ade0a0e7648fa96ee8eaea54140d8), Trace(trace_id=e3b8d7cbcc6d4108beca1e75017e5858), Trace(trace_id=dcb0b08cf5fd4fae99991d02ec6f32f7), Trace(trace_id=b0d65b41e3794246bf248c84a74ba626), Trace(trace_id=4c059314e56a4c16b1368926b3e6140f), Trace(trace_id=0b12c0da8baa4481999fce37a55c091d), Trace(trace_id=344a6a42ad2c46cfbf233f712654012e), Trace(trace_id=c4ab79358f7e43bcaf1783efa6714fb6), Trace(trace_id=4a365a88b4ff4ce69040075526b17818)]

In [51]:
evaluator = Evaluate(
    metric=dspy.evaluate.answer_exact_match,
    devset=valset,
    display_table=True,
    display_progress=True,
    num_threads=8,
)

In [52]:
original_score = evaluator(react)
original_score

Average Metric: 6.00 / 27 (22.2%):  27%|██▋       | 27/100 [05:49<16:00, 13.16s/it]

2025/07/24 05:35:34 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 28.00 / 100 (28.0%): 100%|██████████| 100/100 [18:44<00:00, 11.24s/it]

2025/07/24 05:48:14 INFO dspy.evaluate.evaluate: Average Metric: 28 / 100 (28.0%)


,question,example_answer,trajectory,reasoning,pred_answer,answer_exact_match
0,"What movie did ""the king of cool"" play in with Bud Ekins as his st...","""The Great Escape""","{'thought_0': 'I need to find out which movie ""the king of cool"" p...","The question asks about a movie where ""the king of cool"" played in...","I couldn't find a movie where ""the king of cool"" (Steve McQueen) p...",
1,whos family had their own reality tv show. Robert Kardashian or Ma...,their family reality television series,{'thought_0': 'I need to find out which of these two individuals h...,"Based on the information from both Wikipedia articles, it is clear...",Rob Kardashian,
2,Which star in Shadows in Paradise is a Russian ballerina?,Sofya Skya,{'thought_0': 'I need to find information about the movie Shadows ...,The question asks which star in Shadows in Paradise is a Russian b...,Kati Outinen is not a Russian ballerina.,
3,What was the meaning of the name of the man who appointed Amashsai?,comforter,{'thought_0': 'I need to find out who appointed Amashsai. Let me s...,The question asks about the meaning of the name of the man who app...,"The man who appointed Amashsai was Nehemiah, who was the governor ...",
4,"In addition to the Austrian passport, what is needed to gain acces...",national identity card,"{'thought_0': 'To gain access to 173 countries and territories, I ...",The question asks what is needed in addition to an Austrian passpo...,It appears that an Austrian passport is sufficient to gain access ...,
...,...,...,...,...,...,...
95,"What date did the American actress and singer-songwriter, known fo...","April 19, 1994","{'thought_0': ""I need to find information about an American actres...","To find the release date of Katey Sagal's first album, I started b...","April 19, 1994",✔️ [True]
96,What animated creatures were the title characters of the film whic...,seals,{'thought_0': 'We need to identify the play written by William Sha...,The question asks about the title characters of a film based on on...,The title characters of the film are the fairies Oberon and Titani...,
97,The 1925 Saint Mary's Gaels football team represented what private...,Saint Mary's College of California,"{'thought_0': ""I need to find information about the 1925 Saint Mar...","The 1925 Saint Mary's Gaels football team represented a private, c...",Saint Mary's College of California,✔️ [True]
98,Were Dorothy Arzner and Richard Wallace both French film directors?,no,"{'thought_0': ""I need to verify if Dorothy Arzner and Richard Wall...","Based on the observations from Wikipedia searches, Dorothy Arzner ...","No, neither Dorothy Arzner nor Richard Wallace were French film di...",


🏃 View run eval at: http://localhost:5000/#/experiments/1/runs/7788bea6fddf4c5d95c0db82f7c07b13
🧪 View experiment at: http://localhost:5000/#/experiments/1


28.0

[Trace(trace_id=5ff548d80380486d837d30c98abfeb4a), Trace(trace_id=fafc36305b2748e2adba10f512efde6f), Trace(trace_id=54d4e415f35e423a9b8f8c63086f069e), Trace(trace_id=91c96c841a4049dd913f44555914d8e8), Trace(trace_id=86b3fee6b6dc45698e0f75ff82339e6f), Trace(trace_id=7287b2170ac241afbc20a7ae1440121e), Trace(trace_id=82d1073ae0cb49d0b9257a013b4988cd), Trace(trace_id=3d259b80dd904ff48045bcdd77511a99), Trace(trace_id=6e8fa194cc254965a83631d0cd3bb0a1), Trace(trace_id=f274699ea9eb4fcb86a38a4350bf1d93)]

In [53]:
optimized_score = evaluator(optimized_react)
optimized_score

Average Metric: 43.00 / 100 (43.0%): 100%|██████████| 100/100 [39:50<00:00, 23.91s/it]

2025/07/24 06:28:05 INFO dspy.evaluate.evaluate: Average Metric: 43 / 100 (43.0%)


,question,example_answer,trajectory,reasoning,pred_answer,answer_exact_match
0,"What movie did ""the king of cool"" play in with Bud Ekins as his st...","""The Great Escape""","{'thought_0': 'I need to find out which movie ""the king of cool"" p...","The information about which movie ""the king of cool"" played in wit...",The Great Escape,✔️ [True]
1,whos family had their own reality tv show. Robert Kardashian or Ma...,their family reality television series,{'thought_0': 'I need to find out which family had their own reali...,The information about which family had their own reality TV show c...,Kardashian,
2,Which star in Shadows in Paradise is a Russian ballerina?,Sofya Skya,{'thought_0': 'I need to find out which star in Shadows in Paradis...,The information about which star in Shadows in Paradise is a Russi...,Kati Outinen,
3,What was the meaning of the name of the man who appointed Amashsai?,comforter,"{'thought_0': 'I need to find out who Amashsai was appointed by.',...",The information about who appointed Amashsai can be found in the W...,Nehemiah,
4,"In addition to the Austrian passport, what is needed to gain acces...",national identity card,"{'thought_0': 'To gain access to 173 countries and territories, I ...",To gain access to 173 countries and territories in addition to an ...,Visa-free or visa on arrival status,
...,...,...,...,...,...,...
95,"What date did the American actress and singer-songwriter, known fo...","April 19, 1994",{'thought_0': 'I need to find out what date the American actress a...,The information about Katey Sagal's first album release date can b...,"April 19, 1994",✔️ [True]
96,What animated creatures were the title characters of the film whic...,seals,"{'thought_0': ""I need to find out which animated creatures were th...",The information about which animated creatures were the title char...,Romeo and Juliet (the title characters),
97,The 1925 Saint Mary's Gaels football team represented what private...,Saint Mary's College of California,"{'thought_0': ""I need to find out which private, coeducational col...","The information about which private, coeducational college in Mora...",Saint Mary's College of California,✔️ [True]
98,Were Dorothy Arzner and Richard Wallace both French film directors?,no,{'thought_0': 'I need to find out if Dorothy Arzner and Richard Wa...,The question asks whether Dorothy Arzner and Richard Wallace were ...,We do not have enough information to determine if Dorothy Arzner a...,


🏃 View run eval at: http://localhost:5000/#/experiments/1/runs/d9d5eae7f0184aa1bc2537bc36a28c27
🧪 View experiment at: http://localhost:5000/#/experiments/1


43.0

[Trace(trace_id=f4d76a7a265b41649a38ff0710cf3930), Trace(trace_id=6ec2bab85f934283a3ce0452a53d3604), Trace(trace_id=7f4e55a28a9b45319aadb8d065a42214), Trace(trace_id=8df0336aa3ab4e4a9c4605f2623337b6), Trace(trace_id=57cc7219f8484919904f9f3e10452bdd), Trace(trace_id=59c8153cbbad473193be5cc38779a3a4), Trace(trace_id=e2e879cc25c041e7ac07936ae46b4746), Trace(trace_id=1d391c529cb24a1bbafefa5e0083f363), Trace(trace_id=5e015e8ecd4e49f8974c4c313f54906e), Trace(trace_id=fdba65a3cb064a38a47cd9569ec81a38)]

### llama3.2:3b

In [55]:
lm = LM(
    "ollama_chat/llama3.2:3b",
    api_base="http://localhost:11434",
    api_key="",
    max_tokens=40960,
    temperature=0.0,
    cache=False,
)
dspy.configure(lm=lm)

In [56]:
react = ReAct("question -> answer", tools=[search_wikipedia])

In [57]:
tp = MIPROv2(
    metric=dspy.evaluate.answer_exact_match,
    auto="light",
    num_threads=8,
)

In [58]:
optimized_react = tp.compile(
    react,
    trainset=trainset[:10],
    valset=valset[:10],
    requires_permission_to_run=False,
)

2025/07/24 09:42:38 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '732f35784d714f04a0da85390cc3620c', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current dspy workflow
2025/07/24 09:42:38 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 20
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 10

2025/07/24 09:42:38 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/07/24 09:42:38 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/07/24 09:42:38 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


Bootstrapping set 1/6
Bootstrapping set 2/6
Bootstrapping set 3/6


100%|██████████| 10/10 [01:34<00:00,  9.42s/it]

Bootstrapped 2 full traces after 9 examples for up to 1 rounds, amounting to 10 attempts.


Bootstrapping set 4/6


...


🏃 View run eval_full_19 at: http://localhost:5000/#/experiments/1/runs/228211b7f20b48e387ab2ab012fd2c6b
🧪 View experiment at: http://localhost:5000/#/experiments/1
Average Metric: 0.00 / 10 (0.0%): 100%|██████████| 10/10 [01:14<00:00,  7.48s/it]

2025/07/24 10:24:38 INFO dspy.evaluate.evaluate: Average Metric: 0 / 10 (0.0%)
2025/07/24 10:24:38 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 0.0 with parameters ['Predictor 0: Instruction 1', 'Predictor 0: Few-Shot Set 1', 'Predictor 1: Instruction 1', 'Predictor 1: Few-Shot Set 5'].
2025/07/24 10:24:38 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [0.0, 0.0, 10.0, 0.0, 20.0, 0.0, 0.0, 20.0, 0.0, 10.0, 0.0, 0.0, 20.0, 20.0, 10.0, 0.0, 20.0, 10.0, 20.0, 10.0, 0.0]
2025/07/24 10:24:38 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 20.0
2025/07/24 10:24:38 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2025/07/24 10:24:38 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 20.0!



🏃 View run eval_full_20 at: http://localhost:5000/#/experiments/1/runs/644839495b4a4eb0a236ecd164939b78
🧪 View experiment at: http://localhost:5000/#/experiments/1


🏃 View run casual-owl-302 at: http://localhost:5000/#/experiments/1/runs/732f35784d714f04a0da85390cc3620c
🧪 View experiment at: http://localhost:5000/#/experiments/1


[Trace(trace_id=b96749ae2d8943e78f9a92d3b690a4c4), Trace(trace_id=3b9def2dbf614f7ea5a32026ccf2ff0f), Trace(trace_id=1bd04a0225244409aeed4034cc48648c), Trace(trace_id=281c1e4f8e5b4c2089435333922859f9), Trace(trace_id=e6111aef1a1d42c4bf9c52caf8392a06), Trace(trace_id=0a55821f34c24936aa8f464462273d19), Trace(trace_id=e836dfb48d0e420aba7cf5738deadc45), Trace(trace_id=ae2ccd98acbb42a5bd6accadbc6e4065), Trace(trace_id=b91cd6b6f4ef4a23842806c9e3a1a24b), Trace(trace_id=ee95079dde2d40b791401a12a07e7a12)]

In [59]:
evaluator = Evaluate(
    metric=dspy.evaluate.answer_exact_match,
    devset=valset,
    display_table=True,
    display_progress=True,
    num_threads=8,
)

In [60]:
original_score = evaluator(react)
original_score

Average Metric: 5.00 / 100 (5.0%): 100%|██████████| 100/100 [15:16<00:00,  9.16s/it]

2025/07/24 10:39:55 INFO dspy.evaluate.evaluate: Average Metric: 5 / 100 (5.0%)


,question,example_answer,trajectory,reasoning,pred_answer,answer_exact_match
0,"What movie did ""the king of cool"" play in with Bud Ekins as his st...","""The Great Escape""","{'thought_0': 'The actor who played ""the king of cool"" is Johnny K...","Johnny Knoxville worked with Bud Ekins in several films, but one n...","The movie where Johnny Knoxville played as his stunt double was ""J...",
1,whos family had their own reality tv show. Robert Kardashian or Ma...,their family reality television series,{'thought_0': 'The family of Manvel Gamburyan had their own realit...,Manvel Gamburyan's family had their own reality TV show.,Manvel Gamburyan's family had their own reality TV show.,
2,Which star in Shadows in Paradise is a Russian ballerina?,Sofya Skya,{'thought_0': 'The character Natalia is a Russian ballerina in Sha...,The character Natalia was correctly identified as a Russian baller...,Natalia is a Russian ballerina in Shadows in Paradise.,
3,What was the meaning of the name of the man who appointed Amashsai?,comforter,"{'thought_0': ""The question seems to refer to Amaziah, not Amashsa...","The question refers to Amaziah, not Amashsai. The search results f...",Amaziah was a king of Judah who took the throne at the age of 25 a...,
4,"In addition to the Austrian passport, what is needed to gain acces...",national identity card,{'thought_0': 'Visa requirements for travel to 173 countries and t...,The error occurred because the 'type' argument in the search_wikip...,"To gain access to 173 countries and territories, one needs a visa ...",
...,...,...,...,...,...,...
95,"What date did the American actress and singer-songwriter, known fo...","April 19, 1994",{'thought_0': 'The American actress and singer-songwriter known fo...,The American actress and singer-songwriter known for her role as P...,Florence Henderson's first album release date was not specified in...,
96,What animated creatures were the title characters of the film whic...,seals,"{'thought_0': 'The film is likely ""A Midsummer Night\'s Dream"" sin...","The film is likely ""A Midsummer Night's Dream"" since it is one of ...",The title characters of the film are the four young Athenian lover...,
97,The 1925 Saint Mary's Gaels football team represented what private...,Saint Mary's College of California,"{'thought_0': ""The 1925 Saint Mary's Gaels football team likely re...",The 1925 Saint Mary's Gaels football team likely represented Saint...,Saint Mary's College of California,✔️ [True]
98,Were Dorothy Arzner and Richard Wallace both French film directors?,no,{'thought_0': 'Dorothy Arzner and Richard Wallace were both Americ...,Dorothy Arzner and Richard Wallace were both American film directo...,They were both American film directors.,


🏃 View run eval at: http://localhost:5000/#/experiments/1/runs/9fbf22a8817d498e8b72c8666ebbc68a
🧪 View experiment at: http://localhost:5000/#/experiments/1


5.0

[Trace(trace_id=384dd3e3a5ad4b42b4b278d71e07c0e1), Trace(trace_id=008bec749ae84fabbc72b25ffd2502b9), Trace(trace_id=4da3f62cdb0b436fbe0c70819191b825), Trace(trace_id=f30172430019481bb089b671d7d44d13), Trace(trace_id=fb5919fefeb649d190aa1b119297d151), Trace(trace_id=321df61df0264eac9771a48e02876b98), Trace(trace_id=22194f832e534cc1a10a09cc1ae6f87c), Trace(trace_id=2bf28adb953f4cf89f07bb4f64a9a2d0), Trace(trace_id=d88aa011651647f99eaa36296b48cb90), Trace(trace_id=c11683ca807545a59cf575feee4d4f55)]

In [61]:
optimized_score = evaluator(optimized_react)
optimized_score

Average Metric: 14.00 / 100 (14.0%): 100%|██████████| 100/100 [25:13<00:00, 15.14s/it]

2025/07/24 11:05:09 INFO dspy.evaluate.evaluate: Average Metric: 14 / 100 (14.0%)


,question,example_answer,trajectory,reasoning,pred_answer,answer_exact_match
0,"What movie did ""the king of cool"" play in with Bud Ekins as his st...","""The Great Escape""","{'thought_0': 'Harold Sakata played in ""The King of Cool"" movie wi...","The provided information suggests that ""The King of Cool"" is a nic...",no,
1,whos family had their own reality tv show. Robert Kardashian or Ma...,their family reality television series,"{'thought_0': ""Robert Kardashian's family had their own reality TV...",The Kardashian family had their own reality TV show.,no,
2,Which star in Shadows in Paradise is a Russian ballerina?,Sofya Skya,{'thought_0': 'The star in Shadows in Paradise who is a Russian ba...,"The tool used to search for information is Wikipedia, and it retur...",Natalia Makarova,
3,What was the meaning of the name of the man who appointed Amashsai?,comforter,{'thought_0': 'The name of the man who appointed Amashsai is King ...,The question about the name of the man who appointed Amashsai can ...,King David,
4,"In addition to the Austrian passport, what is needed to gain acces...",national identity card,{'thought_0': 'A valid passport from an International Organization...,A valid passport from an International Organization is needed to g...,no,
...,...,...,...,...,...,...
95,"What date did the American actress and singer-songwriter, known fo...","April 19, 1994","{'thought_0': 'The American actress and singer-songwriter, known f...","The American actress and singer-songwriter, known for her role as ...",no,
96,What animated creatures were the title characters of the film whic...,seals,"{'thought_0': ""The title characters of the film based on Shakespea...",The title characters of the film based on Shakespeare's play are f...,fairies,
97,The 1925 Saint Mary's Gaels football team represented what private...,Saint Mary's College of California,"{'thought_0': ""The 1925 Saint Mary's Gaels football team represent...",The 1925 Saint Mary's Gaels football team represented Saint Mary's...,Saint Mary's College of California,✔️ [True]
98,Were Dorothy Arzner and Richard Wallace both French film directors?,no,"{'thought_0': 'No, Dorothy Arzner was an American film director, w...","Dorothy Arzner was an American film director, while Richard Wallac...",no,✔️ [True]


🏃 View run eval at: http://localhost:5000/#/experiments/1/runs/9c00d178a4e94640afe8f88936d04ed7
🧪 View experiment at: http://localhost:5000/#/experiments/1


14.0

[Trace(trace_id=1ae15f338e2a469195944332e1cdf619), Trace(trace_id=4d81f3af174149a99231a2ea3d918fd6), Trace(trace_id=2d76b02553aa40eea9be3afa6c41361f), Trace(trace_id=492bba6658bc4d1fb2a2a4b452302141), Trace(trace_id=de0eb8b24e2044fdaad581fa4ad394b7), Trace(trace_id=568542d739f4474ca71fdfb0639e4f6d), Trace(trace_id=fa53e0a7dc47495593f9f64cfe1f0416), Trace(trace_id=a3bf7e561e8c412382e900feee12a44d), Trace(trace_id=24b7bb5d018a4a7181f2fd7e9418e517), Trace(trace_id=95b6bf7b0ff44c6e822b2ff9bb6900da)]